# Project 1 — Heart Disease Prediction
Binary classification on the BRFSS 2015 dataset (`_MICHD`).

**Pipeline:**
1. Load stateless-cleaned data
2. fit/transform inside each CV fold (median, NaN flags, one-hot, scaling)
3. Oversample positives + tune threshold
4. Train regularized logistic regression
5. Evaluate (AUC, F1, confusion matrix)
6. Generate submission

In [12]:
import numpy as np
from helpers import create_csv_submission
from implementations import reg_logistic_regression, logistic_regression, ROC_AUC
from preprocess import stateless_clean
from learned_preprocess import fit, transform

## 1. Load data

In [13]:
CLEAN_DIR = "data/clean/"

# Load the stateless-cleaned arrays saved by data_cleaning.ipynb
x_tr_c    = np.load(CLEAN_DIR + "x_train_clean.npy")
x_te_c    = np.load(CLEAN_DIR + "x_test_clean.npy")
y_train   = np.load(CLEAN_DIR + "y_train.npy")      # -1 / +1
train_ids = np.load(CLEAN_DIR + "train_ids.npy")
test_ids  = np.load(CLEAN_DIR + "test_ids.npy")

with open(CLEAN_DIR + "columns.txt") as f:
    clean_names = f.read().splitlines()

y01 = (y_train == 1).astype(float)   # 0/1 labels for logistic loss

n_pos = (y_train ==  1).sum()
n_neg = (y_train == -1).sum()
print(f"x_train: {x_tr_c.shape}  x_test: {x_te_c.shape}")
print(f"Positives: {n_pos}  Negatives: {n_neg}  ratio: {n_neg/n_pos:.1f}x")

x_train: (328135, 275)  x_test: (109379, 275)
Positives: 28975  Negatives: 299160  ratio: 10.3x


## 2. Helpers

In [14]:
def oversample_positives(x, y, seed=42):
    """Duplicate positive samples until classes are balanced."""
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_repeat  = len(neg_idx) // len(pos_idx)
    remainder = len(neg_idx) %  len(pos_idx)
    extra = rng.choice(pos_idx, remainder, replace=False)
    idx = rng.permutation(np.concatenate([neg_idx, np.tile(pos_idx, n_repeat), extra]))
    return x[idx], y[idx]

def f1_score(y_true, y_pred):
    """F1 for binary 0/1 labels."""
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    if tp == 0:
        return 0.0
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    return 2 * p * r / (p + r)

def best_threshold(y_true, scores, thresholds=np.arange(0.1, 0.9, 0.02)):
    """Threshold that maximises F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (scores >= t).astype(float))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))

## 3. Cross-validation
`fit`/`transform` runs inside each fold so no validation data leaks into preprocessing.

In [15]:
def cross_validate(train_fn, x_clean, names, y, k=5, seed=42):
    """
    k-fold CV with proper fit/transform per fold.
    train_fn(y_tr, X_tr) → (w, loss)
    """
    n = len(y)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    folds = np.array_split(idx, k)

    aucs, f1s, thresholds = [], [], []
    for i in range(k):
        val_idx = folds[i]
        tr_idx  = np.concatenate([folds[j] for j in range(k) if j != i])

        # Fit preprocessing on training fold only
        params = fit(x_clean[tr_idx], names)
        X_tr, _ = transform(x_clean[tr_idx], params)
        X_va, _ = transform(x_clean[val_idx], params)

        # Oversample positives in training fold
        X_tr_bal, y_tr_bal = oversample_positives(X_tr, y[tr_idx], seed=seed+i)

        w, _ = train_fn(y_tr_bal, X_tr_bal)

        score = sigmoid(X_va.dot(w))
        _, _, auc = ROC_AUC(y[val_idx], score)
        t, f1 = best_threshold(y[val_idx], score)

        aucs.append(auc)
        f1s.append(f1)
        thresholds.append(t)
        print(f"  fold {i+1}: AUC={auc:.4f}  F1={f1:.4f}  thr={t:.2f}")

    print(f"  ── mean: AUC={np.mean(aucs):.4f}  F1={np.mean(f1s):.4f}  thr={np.mean(thresholds):.2f}")
    return np.mean(aucs), np.mean(f1s), np.mean(thresholds)

def cross_validate(train_fn, x_clean, names, y, k=3, seed=42, subsample=50000):
    """
    k-fold CV with proper fit/transform per fold.
    Uses a subsample for speed during CV — final model trains on full data.
    train_fn(y_tr, X_tr) → (w, loss)
    """
    n = len(y)
    rng = np.random.default_rng(seed)

    # Subsample for CV only (keeps class ratio)
    if subsample and subsample < n:
        pos = np.where(y == 1)[0]
        neg = np.where(y == 0)[0]
        n_pos = int(subsample * len(pos) / n)
        n_neg = subsample - n_pos
        sub_idx = np.concatenate([
            rng.choice(pos, min(n_pos, len(pos)), replace=False),
            rng.choice(neg, min(n_neg, len(neg)), replace=False),
        ])
        sub_idx = rng.permutation(sub_idx)
        x_clean, y = x_clean[sub_idx], y[sub_idx]
        n = len(y)
        print(f"  CV subsample: {n} rows  pos={y.sum():.0f}")

    idx = rng.permutation(n)
    folds = np.array_split(idx, k)

    aucs, f1s, thresholds = [], [], []
    for i in range(k):
        val_idx = folds[i]
        tr_idx  = np.concatenate([folds[j] for j in range(k) if j != i])

        params = fit(x_clean[tr_idx], names)
        X_tr, _ = transform(x_clean[tr_idx], params)
        X_va, _ = transform(x_clean[val_idx], params)

        X_tr_bal, y_tr_bal = oversample_positives(X_tr, y[tr_idx], seed=seed+i)

        w, _ = train_fn(y_tr_bal, X_tr_bal)

        score = sigmoid(X_va.dot(w))
        _, _, auc = ROC_AUC(y[val_idx], score)
        t, f1 = best_threshold(y[val_idx], score)

        aucs.append(auc)
        f1s.append(f1)
        thresholds.append(t)
        print(f"  fold {i+1}: AUC={auc:.4f}  F1={f1:.4f}  thr={t:.2f}")

    print(f"  ── mean: AUC={np.mean(aucs):.4f}  F1={np.mean(f1s):.4f}  thr={np.mean(thresholds):.2f}")
    return np.mean(aucs), np.mean(f1s), np.mean(thresholds)

In [ ]:
MAX_ITERS = 300
GAMMA     = 0.1

def train_logistic(y, X):
    return logistic_regression(y, X, np.zeros(X.shape[1]), MAX_ITERS, GAMMA)

print("Logistic Regression:")
auc, f1, thr = cross_validate(train_logistic, x_tr_c, clean_names, y01)

## 5. Regularized Logistic Regression

In [ ]:
LAMBDA = 1e-4

def train_reg_logistic(y, X):
    return reg_logistic_regression(y, X, LAMBDA, np.zeros(X.shape[1]), MAX_ITERS, GAMMA)

print("Regularized Logistic Regression:")
auc, f1, thr = cross_validate(train_reg_logistic, x_tr_c, clean_names, y01)

## 6. Tune lambda

In [ ]:
lambdas = [1e-5, 1e-4, 1e-3, 1e-2]

results = []
for lam in lambdas:
    def train_fn(y, X, lam=lam):
        return reg_logistic_regression(y, X, lam, np.zeros(X.shape[1]), MAX_ITERS, GAMMA)
    print(f"\nlambda={lam:.0e}:")
    auc, f1, thr = cross_validate(train_fn, x_tr_c, clean_names, y01)
    results.append((lam, auc, f1, thr))

best = max(results, key=lambda r: r[2])
BEST_LAMBDA, BEST_THRESHOLD = best[0], best[3]
print(f"\nBest lambda={BEST_LAMBDA:.0e}  F1={best[2]:.4f}  threshold={BEST_THRESHOLD:.2f}")

## 7. Train final model on full training set

In [ ]:
# Fit preprocessing on the full training set
params_final = fit(x_tr_c, clean_names)
X_train_final, feat_names = transform(x_tr_c, params_final)
X_test_final,  _          = transform(x_te_c, params_final)

print(f"Features after fit/transform: {X_train_final.shape[1]}")

# Oversample on full training set
X_bal, y_bal = oversample_positives(X_train_final, y01)
print(f"Balanced: {X_bal.shape}  pos={y_bal.sum():.0f}  neg={(y_bal==0).sum():.0f}")

w_final, final_loss = reg_logistic_regression(
    y_bal, X_bal, BEST_LAMBDA, np.zeros(X_bal.shape[1]), MAX_ITERS, GAMMA
)

train_score = sigmoid(X_train_final.dot(w_final))
train_f1    = f1_score(y01, (train_score >= BEST_THRESHOLD).astype(float))
_, _, train_auc = ROC_AUC(y01, train_score)

print(f"\nFinal loss : {final_loss:.4f}")
print(f"Train AUC  : {train_auc:.4f}")
print(f"Train F1   : {train_f1:.4f}  (threshold={BEST_THRESHOLD:.2f})")

## 8. Confusion matrix

In [ ]:
pred = (train_score >= BEST_THRESHOLD).astype(float)

tp = int(((pred == 1) & (y01 == 1)).sum())
fp = int(((pred == 1) & (y01 == 0)).sum())
fn = int(((pred == 0) & (y01 == 1)).sum())
tn = int(((pred == 0) & (y01 == 0)).sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

print("Confusion matrix (train):")
print(f"                  Predicted healthy   Predicted sick")
print(f"Actually healthy       {tn:>10d}       {fp:>10d}   ← false alarms")
print(f"Actually sick          {fn:>10d}       {tp:>10d}   ← caught")
print()
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1        : {train_f1:.3f}")
print(f"AUC       : {train_auc:.3f}")

## 9. Generate submission

In [ ]:
test_score = sigmoid(X_test_final.dot(w_final))
y_pred = np.where(test_score >= BEST_THRESHOLD, 1, -1)

print(f"Predicted +1: {(y_pred ==  1).sum()}   Predicted -1: {(y_pred == -1).sum()}")
print(f"Positive rate: {(y_pred == 1).mean()*100:.1f}%")

create_csv_submission(test_ids, y_pred, "submission.csv")
print("Saved → submission.csv")

In [17]:

import numpy as np
from helpers import create_csv_submission
from implementations import reg_logistic_regression, logistic_regression, ROC_AUC
from preprocess import stateless_clean
from learned_preprocess import fit, transform

CLEAN_DIR = "data/clean/"
x_tr_c    = np.load(CLEAN_DIR + "x_train_clean.npy")
x_te_c    = np.load(CLEAN_DIR + "x_test_clean.npy")
y_train   = np.load(CLEAN_DIR + "y_train.npy")
train_ids = np.load(CLEAN_DIR + "train_ids.npy")
test_ids  = np.load(CLEAN_DIR + "test_ids.npy")

with open(CLEAN_DIR + "columns.txt") as f:
    clean_names = f.read().splitlines()

y01 = (y_train == 1).astype(float)
n_pos = (y_train ==  1).sum()
n_neg = (y_train == -1).sum()
print(f"x_train: {x_tr_c.shape}  x_test: {x_te_c.shape}")
print(f"Positives: {n_pos}  Negatives: {n_neg}  ratio: {n_neg/n_pos:.1f}x")


x_train: (328135, 275)  x_test: (109379, 275)
Positives: 28975  Negatives: 299160  ratio: 10.3x


In [18]:

def oversample_positives(x, y, seed=42):
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_repeat  = len(neg_idx) // len(pos_idx)
    remainder = len(neg_idx) %  len(pos_idx)
    extra = rng.choice(pos_idx, remainder, replace=False)
    idx = rng.permutation(np.concatenate([neg_idx, np.tile(pos_idx, n_repeat), extra]))
    return x[idx], y[idx]

def f1_score(y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    if tp == 0: return 0.0
    p = tp / (tp + fp); r = tp / (tp + fn)
    return 2 * p * r / (p + r)

def best_threshold(y_true, scores, thresholds=np.arange(0.1, 0.9, 0.02)):
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (scores >= t).astype(float))
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1

def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))

def cross_validate(train_fn, x_clean, names, y, k=3, seed=42, subsample=50000):
    n = len(y)
    rng = np.random.default_rng(seed)
    if subsample and subsample < n:
        pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
        n_pos = int(subsample * len(pos) / n)
        n_neg = subsample - n_pos
        sub_idx = rng.permutation(np.concatenate([
            rng.choice(pos, min(n_pos, len(pos)), replace=False),
            rng.choice(neg, min(n_neg, len(neg)), replace=False),
        ]))
        x_clean, y = x_clean[sub_idx], y[sub_idx]
        print(f"  CV subsample: {len(y)} rows  pos={y.sum():.0f}")
    idx = rng.permutation(len(y))
    folds = np.array_split(idx, k)
    aucs, f1s, thresholds = [], [], []
    for i in range(k):
        val_idx = folds[i]
        tr_idx  = np.concatenate([folds[j] for j in range(k) if j != i])
        params = fit(x_clean[tr_idx], names)
        X_tr, _ = transform(x_clean[tr_idx], params)
        X_va, _ = transform(x_clean[val_idx], params)
        X_tr_bal, y_tr_bal = oversample_positives(X_tr, y[tr_idx], seed=seed+i)
        w, _ = train_fn(y_tr_bal, X_tr_bal)
        score = sigmoid(X_va.dot(w))
        _, _, auc = ROC_AUC(y[val_idx], score)
        t, f1 = best_threshold(y[val_idx], score)
        aucs.append(auc); f1s.append(f1); thresholds.append(t)
        print(f"  fold {i+1}: AUC={auc:.4f}  F1={f1:.4f}  thr={t:.2f}")
    print(f"  ── mean: AUC={np.mean(aucs):.4f}  F1={np.mean(f1s):.4f}  thr={np.mean(thresholds):.2f}")
    return np.mean(aucs), np.mean(f1s), np.mean(thresholds)

print("Helpers ready.")


Helpers ready.


In [19]:

MAX_ITERS = 300
GAMMA     = 0.1

def train_logistic(y, X):
    return logistic_regression(y, X, np.zeros(X.shape[1]), MAX_ITERS, GAMMA)

print("Logistic Regression:")
auc_lr, f1_lr, thr_lr = cross_validate(train_logistic, x_tr_c, clean_names, y01)


Logistic Regression:
  CV subsample: 50000 rows  pos=4415
  fold 1: AUC=0.8541  F1=0.4181  thr=0.74
  fold 2: AUC=0.8612  F1=0.4219  thr=0.70
  fold 3: AUC=0.8507  F1=0.4103  thr=0.68
  ── mean: AUC=0.8553  F1=0.4168  thr=0.71


In [ ]:

LAMBDA = 1e-4

def train_reg_logistic(y, X):
    return reg_logistic_regression(y, X, LAMBDA, np.zeros(X.shape[1]), MAX_ITERS, GAMMA)

print("Regularized Logistic Regression:")
auc_rl, f1_rl, thr_rl = cross_validate(train_reg_logistic, x_tr_c, clean_names, y01)
